# Benchmark Observations (Live Execution, No Vanilla VLM)

This notebook runs live benchmark execution on 5 cases using lexical retrieval backend only.
It excludes vanilla VLM mode and compares:
- `karcinojen_no_cove_lexical`
- `full_karcinojen_lexical`

If any rate-limit/429 issue appears during execution, it is highlighted so API keys can be switched before rerun.

In [13]:
from pathlib import Path
import json
import sys
import tempfile
import time
from dataclasses import replace
from statistics import mean
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BENCHMARK_FILE = REPO_ROOT / "data" / "mcu-bench" / "benchmark.json"
CONFIG_FILE = REPO_ROOT / "configs" / "model_config.json"

if not BENCHMARK_FILE.exists():
    raise FileNotFoundError(f"Benchmark file not found: {BENCHMARK_FILE}")
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_FILE}")

print(f"Repository root: {REPO_ROOT}")
print(f"Benchmark file: {BENCHMARK_FILE}")
print(f"Config file: {CONFIG_FILE}")

Repository root: E:\help\sem6\genAi\KarcinoJen
Benchmark file: E:\help\sem6\genAi\KarcinoJen\data\mcu-bench\benchmark.json
Config file: E:\help\sem6\genAi\KarcinoJen\configs\model_config.json


In [14]:
records = json.loads(BENCHMARK_FILE.read_text(encoding="utf-8")).get("records", [])
if len(records) < 5:
    raise RuntimeError(f"Need at least 5 benchmark records, found {len(records)}")


def id_key(case_id: str) -> int:
    try:
        return int(case_id.split("_")[-1])
    except Exception:
        return 10**9


records_sorted = sorted(records, key=lambda r: id_key(str(r.get("id", ""))))
selected_records = records_sorted[:5]
selected_case_ids = [r["id"] for r in selected_records]

print("Selected case IDs:", selected_case_ids)
print("Execution modes: ['karcinojen_no_cove_lexical', 'full_karcinojen_lexical']")

Selected case IDs: ['mcu_bench_001', 'mcu_bench_002', 'mcu_bench_003', 'mcu_bench_006', 'mcu_bench_007']
Execution modes: ['karcinojen_no_cove_lexical', 'full_karcinojen_lexical']


In [15]:
from src.extractor.model_config import load_runtime_config
from src.extractor.vlm_client import VLMClient
from src.extractor.vlm_extractor import run_stage5_extraction
from src.extractor.schema_harness import validate_register_extraction
from src.ingest.pdf_page_renderer import render_pdf_page
from src.retrieval.hybrid_retriever import retrieve_top_pages
from src.validator.svd_validator import load_svd_registers, validate_extraction
from src.validator.error_taxonomy import classify_failure
from src.synthesis.synthesize import _synthesize_register, _write_driver_h, _write_driver_c
from src.evaluation.pass_at_k import compute_pass_at_k, evaluate_extraction_against_ground_truth


def to_markdown_table(headers, rows):
    sep = ["---"] * len(headers)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(sep) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    return "\n".join(lines)


def looks_rate_limited(message: str) -> bool:
    text = str(message).lower()
    markers = [
        "429",
        "resource_exhausted",
        "quota exceeded",
        "rate limit",
        "too many requests",
    ]
    return any(marker in text for marker in markers)


def build_page_context(datasheet_path: Path, top_pages: list[dict], render_dir: Path) -> list[dict]:
    page_context = []
    for page in top_pages:
        rendered = render_pdf_page(datasheet_path, int(page["page_number"]), render_dir)
        page_context.append({
            "page_id": page["page_id"],
            "source_file": page["source_file"],
            "page_number": page["page_number"],
            "peripheral": page.get("peripheral", ""),
            "keywords": page.get("keywords", []),
            "page_text": str(page.get("text", ""))[:4000],
            "image_path": str(rendered) if rendered else page.get("image_path"),
        })
    return page_context


def run_single_query_live(*, record: dict, config_name: str, runtime_cfg, enable_validation: bool, enable_cove: bool) -> dict:
    query_id = record["id"]
    query = record["query"]
    ground_truth = record.get("ground_truth") or record.get("ground_truth_json") or {}
    ds_stem = record["datasheet_stem"]
    svd_stem = record["svd_stem"]

    t0 = time.perf_counter()

    def _base_result(**overrides):
        payload = {
            "case_id": query_id,
            "query": query,
            "mode": config_name,
            "status": "FAIL",
            "attempts": 0,
            "schema_valid": False,
            "address_correct": False,
            "pass_at_k": None,
            "cove_recovered": False,
            "elapsed_s": round(time.perf_counter() - t0, 2),
            "rate_limit_hit": False,
            "error": None,
        }
        payload.update(overrides)
        return payload

    try:
        ds_path = REPO_ROOT / "data" / "datasheets" / f"{ds_stem}.pdf"
        svd_path = REPO_ROOT / "data" / "svd" / f"{svd_stem}.svd"

        if not ds_path.exists():
            return _base_result(error=f"Datasheet missing: {ds_path.name}")
        if not svd_path.exists():
            return _base_result(error=f"SVD missing: {svd_path.name}")

        top_pages = retrieve_top_pages(
            query=query,
            datasheet_path=ds_path,
            retrieval_cfg=runtime_cfg.retrieval,
        )
        if not top_pages:
            return _base_result(error="Retrieval returned zero pages")

        with tempfile.TemporaryDirectory() as td:
            render_dir = Path(td) / "rendered"
            page_context = build_page_context(ds_path, top_pages, render_dir)

            registers = load_svd_registers(str(svd_path)) if enable_validation else None
            client = VLMClient(runtime_cfg.provider)

            extraction_cfg = runtime_cfg.extraction
            if not enable_cove:
                extraction_cfg = replace(extraction_cfg, max_attempts=1)

            stage5 = run_stage5_extraction(
                client=client,
                extraction_cfg=extraction_cfg,
                query=query,
                page_context=page_context,
                registers=registers if enable_validation else None,
            )

            extraction = stage5.extraction
            schema_valid = bool(extraction) and validate_register_extraction(extraction).is_valid

            address_correct = False
            pass_at_k = None
            error_category = None

            if extraction is not None:
                gt_eval = evaluate_extraction_against_ground_truth(extraction, ground_truth)
                address_correct = bool(gt_eval.get("address_correct", False))

                if enable_validation and registers:
                    val_result = validate_extraction(extraction, registers)
                    if val_result.status != "PASS":
                        error_category = classify_failure(val_result.checks)

                validated_payload = {
                    **extraction,
                    "validation_status": "PASS",
                    "validation_attempts": len(stage5.attempts),
                    "source_page_id": top_pages[0]["page_id"],
                    "source_file": ds_path.name,
                    "mcu_family": svd_stem.upper(),
                }
                synth = _synthesize_register(validated_payload)
                if synth is not None:
                    out_dir = Path(td) / "synth"
                    out_dir.mkdir(exist_ok=True)
                    h_path = out_dir / "driver.h"
                    c_path = out_dir / "driver.c"
                    _write_driver_h([synth], str(h_path))
                    _write_driver_c([synth], "driver.h", str(c_path))
                    pass_report = compute_pass_at_k(
                        h_path,
                        svd_path,
                        peripheral_filter=record.get("peripheral"),
                    )
                    pass_at_k = float(pass_report.get("accuracy", 0.0))

            attempt_errors = [err for att in stage5.attempts for err in (att.schema_errors or [])]
            rate_limit_hit = any(looks_rate_limited(err) for err in attempt_errors)

            error_message = None
            if stage5.status != "PASS" and attempt_errors:
                error_message = attempt_errors[-1]

            return _base_result(
                status=stage5.status,
                attempts=len(stage5.attempts),
                schema_valid=schema_valid,
                address_correct=address_correct,
                pass_at_k=pass_at_k,
                cove_recovered=bool(enable_cove and len(stage5.attempts) > 1 and stage5.status == "PASS"),
                elapsed_s=round(time.perf_counter() - t0, 2),
                rate_limit_hit=rate_limit_hit,
                error=(error_message or error_category),
            )

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {exc}"
        return _base_result(
            elapsed_s=round(time.perf_counter() - t0, 2),
            rate_limit_hit=looks_rate_limited(error_text),
            error=error_text,
        )

In [ ]:
runtime_cfg = load_runtime_config(CONFIG_FILE)
runtime_cfg_lexical = replace(
    runtime_cfg,
    retrieval=replace(runtime_cfg.retrieval, backend="lexical"),
)

MODE_CONFIGS = [
    ("karcinojen_no_cove_lexical", True, False),
    ("full_karcinojen_lexical", True, True),
]

live_rows = []
rate_limit_events = []

for mode_name, enable_validation, enable_cove in MODE_CONFIGS:
    print(f"\n=== Running mode: {mode_name} (backend=lexical) ===")
    for idx, record in enumerate(selected_records, start=1):
        print(f"[{idx}/{len(selected_records)}] {record['id']} ...", end=" ", flush=True)
        row = run_single_query_live(
            record=record,
            config_name=mode_name,
            runtime_cfg=runtime_cfg_lexical,
            enable_validation=enable_validation,
            enable_cove=enable_cove,
        )
        live_rows.append(row)

        outcome = "OK" if row["status"] == "PASS" else row["status"]
        print(f"{outcome} ({row['elapsed_s']}s)")

        if row["rate_limit_hit"]:
            rate_limit_events.append({
                "mode": mode_name,
                "case_id": row["case_id"],
                "error": row["error"],
            })

        # Gentle pacing to reduce risk of provider-side rate limiting.
        time.sleep(2)

case_headers = [
    "Case ID", "Execution Mode", "Status", "Attempts", "Schema Valid",
    "Address Correct", "PASS@K", "CoVe Recovered", "Elapsed (s)", "Error"
]
case_rows = [
    [
        r["case_id"],
        r["mode"],
        r["status"],
        r["attempts"],
        r["schema_valid"],
        r["address_correct"],
        "N/A" if r["pass_at_k"] is None else f"{r['pass_at_k']:.2f}",
        r["cove_recovered"],
        r["elapsed_s"],
        (str(r["error"])[:120] if r["error"] else ""),
    ]
    for r in live_rows
]

display(Markdown("## Case-Wise Results (Live, Lexical Backend)"))
display(Markdown(to_markdown_table(case_headers, case_rows)))

if rate_limit_events:
    display(Markdown(f"**Rate-limit events detected:** {len(rate_limit_events)}"))
else:
    display(Markdown("**Rate-limit events detected:** 0"))


=== Running mode: karcinojen_no_cove_lexical (backend=lexical) ===
[1/5] mcu_bench_001 ... OK (222.52s)
[2/5] mcu_bench_002 ... OK (29.14s)
[3/5] mcu_bench_003 ... OK (39.71s)
[4/5] mcu_bench_006 ... OK (10.62s)
[5/5] mcu_bench_007 ... OK (16.19s)

=== Running mode: full_karcinojen_lexical (backend=lexical) ===
[1/5] mcu_bench_001 ... 

In [11]:
summary_rows = []
for mode_name, _, _ in MODE_CONFIGS:
    rows = [r for r in live_rows if r["mode"] == mode_name]
    if not rows:
        continue

    pass_rate = mean(1.0 if r["status"] == "PASS" else 0.0 for r in rows)
    schema_valid_rate = mean(1.0 if bool(r["schema_valid"]) else 0.0 for r in rows)
    address_correct_rate = mean(1.0 if bool(r["address_correct"]) else 0.0 for r in rows)
    avg_attempts = mean(float(r["attempts"]) for r in rows)
    avg_elapsed = mean(float(r["elapsed_s"]) for r in rows)

    passk_values = [r["pass_at_k"] for r in rows if r["pass_at_k"] is not None]
    avg_passk = mean(passk_values) if passk_values else None

    cove_values = [r["cove_recovered"] for r in rows if r["cove_recovered"] is not None]
    cove_rate = mean(1.0 if bool(v) else 0.0 for v in cove_values) if cove_values else None

    summary_rows.append([
        mode_name,
        f"{pass_rate:.2f}",
        f"{schema_valid_rate:.2f}",
        f"{address_correct_rate:.2f}",
        f"{avg_passk:.2f}" if avg_passk is not None else "N/A",
        f"{avg_attempts:.2f}",
        f"{avg_elapsed:.2f}",
        f"{cove_rate:.2f}" if cove_rate is not None else "N/A",
    ])

summary_headers = [
    "Execution Mode", "Pass Rate", "Schema-Valid Rate", "Address-Correct Rate",
    "Avg PASS@K", "Avg Attempts", "Avg Elapsed (s)", "CoVe Recovery Rate"
]

display(Markdown("## Aggregate Metrics (Live, Lexical Backend)"))
display(Markdown(to_markdown_table(summary_headers, summary_rows)))

## Aggregate Metrics (Live, Lexical Backend)

| Execution Mode | Pass Rate | Schema-Valid Rate | Address-Correct Rate | Avg PASS@K | Avg Attempts | Avg Elapsed (s) | CoVe Recovery Rate |
| --- | --- | --- | --- | --- | --- | --- | --- |
| karcinojen_no_cove_lexical | 1.00 | 1.00 | 1.00 | 1.00 | 1.40 | 66.09 | 0.00 |
| full_karcinojen_lexical | 1.00 | 1.00 | 0.80 | 0.80 | 1.40 | 109.79 | 0.20 |

In [12]:
observations = [
    "Vanilla VLM has been removed from this notebook; only lexical-backend executions are shown.",
]

if len(summary_rows) >= 2:
    by_mode = {row[0]: row for row in summary_rows}
    no_cove_key = "karcinojen_no_cove_lexical"
    full_key = "full_karcinojen_lexical"

    if no_cove_key in by_mode and full_key in by_mode:
        observations.append(
            "Comparison now focuses on no-CoVe vs full CoVe under the same lexical retrieval backend."
        )

if summary_rows:
    fastest_mode = min(summary_rows, key=lambda r: float(r[6]))[0]
    observations.append(f"Fastest mode on this run: {fastest_mode}.")

if rate_limit_events:
    observations.append(
        "429/rate-limit signals were detected. Switch API key and rerun the execution cell before finalizing numbers."
    )
else:
    observations.append("No 429/rate-limit signals were detected in this execution run.")

display(Markdown("## Observations"))
for idx, text in enumerate(observations, start=1):
    display(Markdown(f"{idx}. {text}"))

if rate_limit_events:
    event_headers = ["Execution Mode", "Case ID", "Error Snippet"]
    event_rows = [
        [e["mode"], e["case_id"], (str(e["error"])[:140] if e["error"] else "")]
        for e in rate_limit_events
    ]
    display(Markdown("## Rate-Limit Event Details"))
    display(Markdown(to_markdown_table(event_headers, event_rows)))

## Observations

1. Vanilla VLM has been removed from this notebook; only lexical-backend executions are shown.

2. Comparison now focuses on no-CoVe vs full CoVe under the same lexical retrieval backend.

3. Fastest mode on this run: karcinojen_no_cove_lexical.

4. No 429/rate-limit signals were detected in this execution run.